# Density Matrix Segregator

This notebook provides the code used to separate density matrices into pure or mixed states, both classically and using quantum machine learning. The language used for this code is Julia.

## Functions to generate density matrices

We import the necessary packages

In [1]:
using LinearAlgebra
using MLJ,MLJLinearModels
using LIBSVM

We now define the functions required to generate pure and mixed state density matrices. Pure state density matrices can be denoted as $\rho=\ket{\psi}\bra{\psi}$ and mixed state density matrices are denoted as $\sum_{i=1}^N p_i\rho_i$, where $\rho_i=\ket{\psi_i}\bra{\psi_i}$

In [2]:
begin
	#Function to generate pure state density matrices
	
	function rhopure(n)
		psi=rand(Float64,n)
		psi=psi/norm(psi)
		rho=psi*psi'
		return rho
	end

	function rhomixed(n)
		r1=rhopure(n)
		r2=rhopure(n)
		p=rand(Float64)
		rho=p*r1+(1-p)*r2
		return rho
	end 
	
end

rhomixed (generic function with 1 method)

In [3]:
begin
	rho=rhomixed(2)
end

2×2 Matrix{Float64}:
 0.708845  0.199696
 0.199696  0.291155

In [4]:
begin
    
#Set dimension for density matrix here
d=2

#Set number of data points here
n=1000

#Setting up the data and label matrix
    
data = [zeros(d,d) for i=1:n]
label= zeros(n)
    
for p in 1:n
    i=rand((0,1))
    if (i==0)
        rho=rhopure(d)
    else
        rho=rhomixed(d)
    end
    data[p]=rho
    label[p]=i
end
    
end

In [13]:
begin

#2 Dimensional Feature space matrix, each contains two quantities, Tr(rho) and Tr(rho*rho)
f2d=[zeros(2) for i=1:n]
for i in 1:n
    f2d[i][1]=tr(data[i])
    f2d[i][2]=tr(data[i]*data[i])
end

#Set the number of points to be kept aside as test data    
test=200

#Split into training and test data.
xtrain=f2d[1:n-test]
ytrain=label[1:n-test]
xtest=f2d[n-test+1:n]
ytest=label[n-test+1:n]
test
end

200

In [10]:
X_train = hcat(xtrain...)  # Shape: (2, n-test)
X_test  = hcat(xtest...)   # Shape: (2, test)

model = svmtrain(X_train, ytrain,kernel = Kernel.Linear)
# Predict on test set
predicted_labels, decision_values = svmpredict(model, X_test)

# Accuracy
accuracy = sum(predicted_labels .== ytest) / length(ytest)
println("Accuracy: $(round(accuracy * 100, digits=2))%")

# Confusion matrix (manual)
tp = sum((predicted_labels .== 1) .& (ytest .== 1))
tn = sum((predicted_labels .== 0) .& (ytest .== 0))
fp = sum((predicted_labels .== 1) .& (ytest .== 0))
fn = sum((predicted_labels .== 0) .& (ytest .== 1))
println("TP=$tp, TN=$tn, FP=$fp, FN=$fn")

Accuracy: 63.0%
TP=42, TN=84, FP=0, FN=74


In [12]:
begin
    
#3 Dimensional Feature space matrix, each contains two quantities, Tr(rho), Tr(rho*rho), Tr (rho^3)
f3d=[zeros(3) for i=1:n]
for i in 1:n
    f3d[i][1]=tr(data[i])
    f3d[i][2]=tr(data[i]*data[i])
    f3d[i][3]=tr(data[i]^3)
end

#Set the number of points to be kept aside as test data    
test=200

#Split into training and test data.
x3train=f3d[1:n-test]
ytrain=label[1:n-test]
x3test=f3d[n-test+1:n]
ytest=label[n-test+1:n]

X_3train = hcat(x3train...)  # Shape: (2, n-test)
X_3test  = hcat(x3test...)   # Shape: (2, test)

model = svmtrain(X_3train, ytrain,kernel = Kernel.Linear)
# Predict on test set
predicted_labels, decision_values = svmpredict(model, X_3test)

# Accuracy
accuracy = sum(predicted_labels .== ytest) / length(ytest)
println("Accuracy: $(round(accuracy * 100, digits=2))%")

# Confusion matrix (manual)
tp = sum((predicted_labels .== 1) .& (ytest .== 1))
tn = sum((predicted_labels .== 0) .& (ytest .== 0))
fp = sum((predicted_labels .== 1) .& (ytest .== 0))
fn = sum((predicted_labels .== 0) .& (ytest .== 1))
println("TP=$tp, TN=$tn, FP=$fp, FN=$fn")
    
end

Accuracy: 70.0%
TP=56, TN=84, FP=0, FN=60


In [8]:
X_3train

3×800 Matrix{Float64}:
 1.0       1.0       1.0  1.0  1.0       …  1.0  1.0       1.0       1.0
 0.999999  0.952948  1.0  1.0  0.662687     1.0  0.967783  0.781649  0.938077
 0.999999  0.929421  1.0  1.0  0.49403      1.0  0.951674  0.672474  0.907115